# EDA on ultimate marathon data

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw/data"

spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
%sql
SHOW SCHEMAS IN marathos;

In [0]:
DATA_PATH = "/Volumes/marathos/default/raw/data"

schema = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{DATA_PATH}/TWO_CENTURIES_OF_UM_RACES.csv")
    .schema
)

schema

In [0]:
df = spark.read.csv(f"{DATA_PATH}/TWO_CENTURIES_OF_UM_RACES.csv", header = True)

In [0]:
df.printSchema()

In [0]:
df.limit(5).display()

## Specify schema for calculations

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Year of event", IntegerType()),
    StructField("Event dates", StringType()),
    StructField("Event name", StringType()),
    StructField("Event distance/length", StringType()),
    StructField("Event number of finishers", IntegerType()),
    StructField("Athlete performance", StringType()),
    StructField("Athlete club", StringType()),
    StructField("Athlete country", StringType()),
    StructField("Athlete year of birth", DoubleType()),
    StructField("Athlete gender", StringType()),
    StructField("Athlete age category", StringType()),
    StructField("Athlete average speed", DoubleType()),
    StructField("Athlete ID", IntegerType())
])

df_ultra_marathon_schema = spark.read.csv(f"{DATA_PATH}/TWO_CENTURIES_OF_UM_RACES.csv", header = True, schema = schema)
display(df_ultra_marathon_schema.take(5))

In [0]:
print(f"Number of rows {df_ultra_marathon_schema.count()}")
print(f"Number of columns {len(df_ultra_marathon_schema.columns)}")

In [0]:
df_ultra_marathon_schema.select(
    [
        column
        for column, type_ in df_ultra_marathon_schema.dtypes
        if type_ in ("int", "bigint", "double", "decimal")
    ]
).summary().display()

## Check null values

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_ultra_marathon_schema.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

## Look at amount of unique events

In [0]:
from pyspark.sql import functions as sf
df_ultra_marathon_schema.select(sf.count_distinct("Event name")).show()

## Most represented countries

In [0]:
df_countries = df_ultra_marathon_schema.groupBy("Athlete country") \
                                       .count() \
                                       .withColumnRenamed("count", "times_represented") \
                                       .orderBy("times_represented", ascending = False)

df_countries.limit(5).display()

## Ages in different ways

In [0]:
# Calculate ages for the specific year of the race

df_age = df_ultra_marathon_schema.withColumn("Age", col("Year of event") - col("Athlete year of birth").cast("float").cast("int"))
df_age.select("Age").summary().display()

In [0]:
# narrowing down ages for more accuracy 

df_age_filter = df_age.filter((col("Age") > 10) & (col("Age") < 90))
df_age_filter.select("Age").summary().display()

In [0]:
# See what ages runs in what age category

df_age_filter.select("Age", "Athlete age category").limit(5).display()